<a href="https://colab.research.google.com/github/mcristinasanchez-ai/TFG_MATEM-TICAS_M.CristinaS-nchezTim-n/blob/main/ABC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
# 1. Funciones objetivo que vamos a minimizar
def sphere(x):
    return np.sum(x**2)

def schwefel_1(x):
    return np.sum(np.abs(x)) + np.prod(np.abs(x))

def schwefel_2(x):
    return np.sum([np.sum(x[:i+1])**2 for i in range(len(x))])

def schwefel_3(x):
    return np.max(np.abs(x))

def rosenbrock(x):
    return np.sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (x[:-1] - 1.0)**2.0)

def step(x):
    return np.sum(np.floor(x + 0.5)**2)

def quartic(x):
    return np.sum(np.arange(1, len(x) + 1) * (x**4)) + np.random.random()

def schwefel_4(x):
    return np.sum(-x * np.sin(np.sqrt(np.abs(x))))

def rastrigin(x):
    return np.sum(x**2 - 10 * np.cos(2 * np.pi * x) + 10)

def ackley(x):
    n = float(len(x))
    return -20 * np.exp(-0.2 * np.sqrt(np.sum(x**2) / n)) - np.exp(np.sum(np.cos(2 * np.pi * x)) / n) + 20 + np.exp(1)

funciones_prueba = {
    "Sphere": (sphere, -100, 100),
    "Schwefel 1": (schwefel_1, -10, 10),
    "Schwefel 2": (schwefel_2, -100, 100),
    "Schwefel 3": (schwefel_3, -100, 100),
    "Rosenbrock": (rosenbrock, -30, 30),
    "Step": (step, -100, 100),
    "Quartic": (quartic, -1.28, 1.28),
    "Schwefel 4": (schwefel_4, -500, 500),
    "Rastrigin": (rastrigin, -5.12, 5.12),
    "Ackley": (ackley, -32, 32)
}

# 2. Parámetros
num_fuentes = 100  #cantidad de fuentes en el espacio de búsqueda
num_iteraciones = 10000 #número de iteraciones
dimension = 30 #espacio de búsqueda de 30 dimensiones
num_ejecuciones = 10  #número de ejecuciones por cada función

# Parámetro principal que nos dcie el número de veces que se busca en
#en una fuente de alimento sin tener éxito
limite_intentos = num_fuentes * dimension

#Se Almacenan los mejores resultados de cada ejecución
resultados_totales = {}

# 3. Función fitness (Se convierte la función objetivo por que en el ABC, se maximiza el fitness)
def calcular_fitness(valor_obj):
    if valor_obj >= 0:
        return 1 / (1 + valor_obj)
    else:
        return 1 + np.abs(valor_obj)


print("Iniciando optimizaciones con ABC...")

for nombre, (f, limite_inf, limite_sup) in funciones_prueba.items():
    print(f"\nEjecutando 10 veces para: {nombre}...")
    resultados_funcion = []  #se almacenan los 10 mejores globales de esta función

    for ejecucion in range(num_ejecuciones):
        fuentes = np.random.uniform(limite_inf, limite_sup, (num_fuentes, dimension)) #matriz donde se colocan las 100 fuentes de forma aleatoria
        valores_obj = np.array([f(sol) for sol in fuentes]) #evaluación inicial de la función objetivo
        fitness = np.array([calcular_fitness(v) for v in valores_obj]) #se calcula el firness asociado a cada fuente de alimento

         # 3. Inicialización
        intentos = np.zeros(num_fuentes)
        gmejor_valor = np.min(valores_obj) #se evalua su valor

        # 4. Bucle principal del algoritmo ABC
        for iter in range(num_iteraciones):

            # FASE EMPLEADAS
            for i in range(num_fuentes):
                k = np.random.choice([idx for idx in range(num_fuentes) if idx != i]) #se selecciona una fuenta k disntina de i
                j = np.random.randint(0, dimension) #se selecciona una fuenta j

                # Se genera una nueva solución a partir de la ecuación de movimiento
                nueva_sol = fuentes[i].copy()
                # Ecuación de movimiento combinando xi,k con xk,j
                nueva_sol[j] += np.random.uniform(-1, 1) * (fuentes[i, j] - fuentes[k, j])

                #Se usa np.clip para que las partículas no busquen  en zonas no válidas
                #si se superan los límites permitidos
                nueva_sol = np.clip(nueva_sol, limite_inf, limite_sup)

                v_nuevo = f(nueva_sol) #se evalua la nueva solución
                fit_nuevo = calcular_fitness(v_nuevo) #se calcula su fitness

                # Si la nueva solución mejora, reemplaza a la fuente actual; en caso contrario se incrementa el contador de intentos
                if fit_nuevo > fitness[i]:
                    fuentes[i], valores_obj[i], fitness[i] = nueva_sol, v_nuevo, fit_nuevo
                    intentos[i] = 0
                else:
                    intentos[i] += 1

            # FASE DE ABEJAS OBSERVADORAS
            # Se calcula la probabilidad de selección de cada fuente
            suma_fitness = np.sum(fitness)
            probabilidades = fitness / suma_fitness if suma_fitness > 0 else np.ones(num_fuentes)/num_fuentes
            probabilidades = probabilidades / probabilidades.sum()

            # Se genera una solución nueva usando la misma ecuación de movimiento que en la fase anterior
            for _ in range(num_fuentes):
                i = np.random.choice(range(num_fuentes), p=probabilidades) #se selecciona la fuente i

                # Solo trabajamos en fuentes que no hayan superado el límite
                if intentos[i] <= limite_intentos:
                    k = np.random.choice([idx for idx in range(num_fuentes) if idx != i]) #se selecciona una fuenta k distina de i
                    j = np.random.randint(0, dimension) #se selecciona una fuenta j

                    nueva_sol = fuentes[i].copy()
                    nueva_sol[j] += np.random.uniform(-1, 1) * (fuentes[i, j] - fuentes[k, j])
                    nueva_sol = np.clip(nueva_sol, limite_inf, limite_sup)

                    v_nuevo = f(nueva_sol)
                    fit_nuevo = calcular_fitness(v_nuevo)

                    if fit_nuevo > fitness[i]:
                        fuentes[i], valores_obj[i], fitness[i] = nueva_sol, v_nuevo, fit_nuevo
                        intentos[i] = 0
                    else:
                        intentos[i] += 1

            # FASE DE ABEJAS EXPLORADORAS
            # Si se supera el límite de intentos la fuente se abandona y se
            #sustituye por una nueva elegida de forma aleatoria
            for i in range(num_fuentes):
                if intentos[i] > limite_intentos:
                    fuentes[i] = np.random.uniform(limite_inf, limite_sup, dimension)
                    valores_obj[i] = f(fuentes[i])
                    fitness[i] = calcular_fitness(valores_obj[i])
                    intentos[i] = 0
           # Se actualiza el mejor valor encontrado
            mejor_actual = np.min(valores_obj)
            if mejor_actual < gmejor_valor:
                gmejor_valor = mejor_actual

        resultados_funcion.append(gmejor_valor)
        print(f"  Ejecución {ejecucion + 1}: {gmejor_valor:.4e}")

    resultados_totales[nombre] = resultados_funcion

#CREAMOS LA TABLA CON LOS RESULTADOS DE CADA FUNCIÓN EN LAS 10 Iteraciones y
#Y CALCULAMOS LA MEDIA, MEDIANA Y VARIANZA

# 1. Se crea el Dataframe donde cada columna es una de las funciones de prueba y
#y cada fila son las ejecuiones
df_inicial = pd.DataFrame(resultados_totales)
df_inicial.index = [f"EJECUCIÓN {i+1}" for i in range(num_ejecuciones)]
df_final = df_inicial.copy()

# 2. Se calcula la media, mediana y varianza
df_final.loc['MEDIA'] = df_inicial.mean(axis=0)
df_final.loc['MEDIANA'] = df_inicial.median(axis=0)
df_final.loc['VARIANZA'] = df_inicial.var(axis=0)

# 3. Se muestan los resultados por pantalla y se guardan los resultados en la
#hoja de cáculo
print("\n" + "="*50)
print("TABLA COMPARATIVA FINAL (ABC)")
print("="*50)
pd.set_option('display.float_format', lambda x: f'{x:.4e}')
print(df_final)
df_final.to_excel("Tabla_comparativa_ABC.xlsx")

Iniciando optimizaciones con ABC...

Ejecutando 10 veces para: Sphere...
  Ejecución 1: 3.1655e-16
  Ejecución 2: 4.5349e-16
  Ejecución 3: 4.2883e-16
  Ejecución 4: 4.0924e-16
  Ejecución 5: 4.7045e-16
  Ejecución 6: 3.1300e-16
  Ejecución 7: 4.5610e-16
  Ejecución 8: 3.1914e-16
  Ejecución 9: 4.2314e-16
  Ejecución 10: 3.0739e-16

Ejecutando 10 veces para: Schwefel 1...
  Ejecución 1: 1.1662e-15
  Ejecución 2: 9.8141e-16
  Ejecución 3: 9.3979e-16
  Ejecución 4: 9.8994e-16
  Ejecución 5: 7.6073e-16
  Ejecución 6: 9.8348e-16
  Ejecución 7: 9.6610e-16
  Ejecución 8: 9.2005e-16
  Ejecución 9: 7.6628e-16
  Ejecución 10: 9.9273e-16

Ejecutando 10 veces para: Schwefel 2...
  Ejecución 1: 6.9903e+02
  Ejecución 2: 3.4411e+02
  Ejecución 3: 4.0389e+02
  Ejecución 4: 1.7491e+02
  Ejecución 5: 2.2871e+02
  Ejecución 6: 9.8923e+02
  Ejecución 7: 3.8523e+02
  Ejecución 8: 2.4384e+02
  Ejecución 9: 3.8342e+02
  Ejecución 10: 5.6755e+02

Ejecutando 10 veces para: Schwefel 3...
  Ejecución 1: 3.4812